# Making It Production-Ready

In the previous notebook we built a 3-agent DevOps pipeline. It works — but it has problems you'd hit immediately in production. This notebook fixes them, one at a time.

| Section | The Problem | The Fix | CrewAI API |
|---------|------------|---------|------------|
| 1 | Agent output is raw text — parsing it is fragile | **Structured Output** | `output_pydantic=Model` |
| 2 | Bad or vague output passes through unchecked | **Code Guardrail** | `guardrail=validate_fn` |
| 3 | Writing validation functions for everything is tedious | **No-Code Guardrail** | `guardrail="plain English"` |

In [2]:
import os
from typing import Any, Tuple

from crewai import Agent, Crew, Process, Task
from crewai.llm import LLM
from crewai.tasks.task_output import TaskOutput
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()

llm = LLM(
    model="openai/gpt-4o-mini",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

In [4]:
LOG_INPUT = """
[2024-01-15 14:32:15.123] INFO: Starting deployment of myapp-deployment
[2024-01-15 14:32:16.567] WARNING: Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state
[2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start
[2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require 'docker login'
[2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff
[2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline
[2024-01-15 14:32:26.789] WARNING: Service myapp-service has no available endpoints
[2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated
"""

log_analyzer = Agent(
    role="DevOps Log Analyzer",
    goal="Analyze log files to identify and extract specific issues, errors, and failure patterns",
    llm=llm,
    backstory="""You are a senior DevOps engineer with 10 years of experience in 
    analyzing production logs and identifying critical issues. You excel at parsing 
    through complex log files, identifying error patterns, extracting relevant error 
    messages, and determining the root cause of failures from log data.""",
    verbose=False,
)

---

## 1. Structured Output

### The Problem: Raw Text Is Fragile

In Notebook 1, our agent returns a wall of markdown text. Let's run it and try to extract specific fields.

In [5]:
v1_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="""A detailed analysis report containing:
    - Primary issue description
    - Key error messages and codes
    - Timeline of failure events
    - Root cause analysis
    - Affected components""",
    agent=log_analyzer,
)

v1_crew = Crew(
    agents=[log_analyzer],
    tasks=[v1_task],
    process=Process.sequential,
    verbose=False,
)

v1_result = await v1_crew.kickoff_async(inputs={"log_data": LOG_INPUT})

In [6]:
print(v1_result.raw)

**Analysis Report for Deployment Issues with myapp-deployment**

**Primary Issue Description:**
The deployment of the application `myapp-deployment` has encountered multiple critical issues leading to a failed production deployment. The primary problem appears to be related to image retrieval, which is preventing the successful start of the associated pods.

**Key Error Messages and Codes:**
1. `WARNING: Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state`
2. `ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start`
3. `ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require 'docker login'`
4. `ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff`
5. `ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline`
6. `WARNING: Service myapp-service has no available endpoints`
7. `CRITICAL: Production deployment failed - rollback initiated`

**Timeline of Failure Events:**
- **2024-01-

In [7]:
# Want the root cause? Parse the string:
lines = v1_result.raw.split("\n")
for line in lines:
    if "root cause" in line.lower():
        print(f"Found: {line}")
        break

# Want the number of errors? Count manually in free-form text?
# Want to pass typed data to the next agent? Impossible.
# What if the agent formats it differently next run? Everything breaks.

Found: **Root Cause Analysis:**


### The Fix: `output_pydantic`

Define a Pydantic model describing the shape of the output you want. Add `output_pydantic=Model` to the task. CrewAI forces the agent to return data matching that schema.

In [8]:
class LogAnalysisReport(BaseModel):
    primary_issue: str = Field(description="One-line description of the main issue")
    root_cause: str = Field(description="Root cause analysis based on log evidence")
    errors: list[str] = Field(description="All errors found in the log")
    affected_components: list[str] = Field(description="System components affected")
    timeline: list[str] = Field(description="Sequence of events leading to failure")

In [9]:
structured_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="A structured log analysis report",
    output_pydantic=LogAnalysisReport,
    agent=log_analyzer,
)

structured_crew = Crew(
    agents=[log_analyzer],
    tasks=[structured_task],
    process=Process.sequential,
    verbose=False,
)


structured_result = await structured_crew.kickoff_async(inputs={"log_data": LOG_INPUT})

In [10]:
report = structured_result.pydantic

print(f"Primary issue: {report.primary_issue}")

Primary issue: Production deployment failed due to image pull errors.


In [11]:
print(f"Root cause: {report.root_cause}")

Root cause: The deployment of myapp failed because the required Docker image 'myapp:v1.2.3' could not be pulled, leading to an ImagePullBackOff status and triggering a rollback.


In [12]:
print(f"\nErrors found: {len(report.errors)}")
for error in report.errors:
    print(f"  - {error}")


Errors found: 7
  - Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state
  - Pod myapp-deployment-7b8c9d5f4-abc12 failed to start
  - Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require 'docker login'
  - Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff
  - Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline
  - Service myapp-service has no available endpoints
  - Production deployment failed - rollback initiated


In [13]:
print(f"\nAffected components: {report.affected_components}")


Affected components: ['Pod myapp-deployment-7b8c9d5f4-abc12', 'Deployment myapp-deployment', 'Service myapp-service']


In [14]:
print(f"\nTimeline:")
for event in report.timeline:
    print(f"  - {event}")


Timeline:
  - 2024-01-15 14:32:15.123: Starting deployment of myapp-deployment
  - 2024-01-15 14:32:16.567: Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state
  - 2024-01-15 14:32:17.890: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start
  - 2024-01-15 14:32:18.123: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require 'docker login'
  - 2024-01-15 14:32:18.456: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff
  - 2024-01-15 14:32:25.901: Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline
  - 2024-01-15 14:32:26.789: Service myapp-service has no available endpoints
  - 2024-01-15 14:32:29.456: Production deployment failed - rollback initiated


In [15]:
print(f"\nFull JSON:\n{report.model_dump_json(indent=2)}")


Full JSON:
{
  "primary_issue": "Production deployment failed due to image pull errors.",
  "root_cause": "The deployment of myapp failed because the required Docker image 'myapp:v1.2.3' could not be pulled, leading to an ImagePullBackOff status and triggering a rollback.",
  "errors": [
    "Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state",
    "Pod myapp-deployment-7b8c9d5f4-abc12 failed to start",
    "Failed to pull image \"myapp:v1.2.3\": pull access denied, repository does not exist or may require 'docker login'",
    "Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff",
    "Deployment rollout failed: deployment \"myapp-deployment\" exceeded its progress deadline",
    "Service myapp-service has no available endpoints",
    "Production deployment failed - rollback initiated"
  ],
  "affected_components": [
    "Pod myapp-deployment-7b8c9d5f4-abc12",
    "Deployment myapp-deployment",
    "Service myapp-service"
  ],
  "timeline": [
    "2024-01-15 14:32:15.123:

Same agent. Same log. One parameter changed everything — `output_pydantic=LogAnalysisReport`. Now every field is typed, accessible, and guaranteed to be there.

---

## 2. Code Guardrail

### The Problem: Bad Output Passes Through

Structured output guarantees the *shape*, but not the *quality*. What if the agent returns a report with zero errors identified? With no guardrail, that bad output flows straight to the next agent unchecked.

Here's a log where everything is labeled INFO — no explicit ERROR lines. The agent sees "no errors" and returns an empty `errors` list. The shape is valid, but the content is useless.

In [16]:
TRICKY_LOG_INPUT = """
[2024-01-15 09:01:22.100] INFO: Cron job scheduled-cleanup started
[2024-01-15 09:01:23.200] INFO: Connected to database cluster (primary)
[2024-01-15 09:01:24.300] INFO: Processing batch 1 of 1
[2024-01-15 09:01:25.400] INFO: 0 records processed
[2024-01-15 09:01:26.500] INFO: Disk usage at 94%
[2024-01-15 09:01:27.600] INFO: Cron job scheduled-cleanup completed successfully
"""

unguarded_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="A structured log analysis report",
    output_pydantic=LogAnalysisReport,
    agent=log_analyzer,
)

unguarded_crew = Crew(
    agents=[log_analyzer],
    tasks=[unguarded_task],
    process=Process.sequential,
    verbose=False,
)

unguarded_result = await unguarded_crew.kickoff_async(inputs={"log_data": TRICKY_LOG_INPUT})

In [17]:
report = unguarded_result.pydantic
print(f"Errors found: {len(report.errors)}")
print(f"Errors: {report.errors}")
print(f"Root cause: {report.root_cause}")

passed = len(report.errors) > 0
print(f"\nWould pass guardrail? {passed}")
if not passed:
    print("The agent saw all-INFO logs and returned zero errors.")
    print("But '0 records processed' and '94% disk' ARE problems.")
    print("Without a guardrail, this empty report goes straight to the next agent.")

Errors found: 0
Errors: []
Root cause: The cron job completed successfully but processed no records, which may indicate data availability issues or previous record deletion.

Would pass guardrail? False
The agent saw all-INFO logs and returned zero errors.
But '0 records processed' and '94% disk' ARE problems.
Without a guardrail, this empty report goes straight to the next agent.


### The Fix: Add a Guardrail

A guardrail is a function that validates the output *before* it's accepted. It returns:
- `(True, data)` — output is good, pass it through
- `(False, "reason")` — output is rejected, agent retries automatically

Now the same log, same agent — but with the guardrail. When the agent returns zero errors, the guardrail rejects it and the agent retries, digging deeper into the INFO messages to find the real issues. Run with `verbose=True` to see the retry in action.

In [18]:
def validate_log_analysis(result: TaskOutput) -> Tuple[bool, Any]:
    report = result.pydantic
    if not report or not report.errors:
        return (False, "Must identify at least one error")
    return (True, report)

In [19]:
guarded_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="A structured log analysis report",
    output_pydantic=LogAnalysisReport,
    guardrail=validate_log_analysis,
    agent=log_analyzer,
)

guarded_crew = Crew(
    agents=[log_analyzer],
    tasks=[guarded_task],
    process=Process.sequential,
    verbose=True,
)

guarded_result = await guarded_crew.kickoff_async(inputs={"log_data": TRICKY_LOG_INPUT})

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.13                                                                                       │
│  Latest version:  1.15.15                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d1e398ff-062d-45ae-8b18-88625d75b3fa                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following log data to identify issues:                                                       │
│                                                                                                                 │
│  [2024-01-15 09:01:22.100] INFO: Cron job scheduled-cleanup started                                             │
│  [2024-01-15 09:01:23.200] INFO: Connected to database cluster (primary)                                        │
│  [2024-01-15 09:01:24.300] INFO: Processing batch 1 of 1                                                        │
│  [2024-01-15 09:01:25.400] INFO: 0 records processed                                                            │
│  [2024-01-15 09:01:26.500] INFO: Disk usage at 94%                                                              │
│  [2024-01-15 09:01:27.600] INFO: Cron job scheduled-cleanup completed successfully                              │
│                                                                                                                 │
│  ID: 8c451e3b-e335-4d5b-8801-005be1b068f8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🛡️ Guardrail Check ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Evaluation Started                                                                                   │
│  Name: def validate_log_analysis(result: TaskOutput) -> T...                                                    │
│  Status: 🔄 Evaluating                                                                                          │
│  Attempt: 1                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🛡️ Guardrail Failed ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Failed                                                                                               │
│  Name: Validation Error                                                                                         │
│  Error: Must identify at least one error                                                                        │
│  Attempts: 1                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🛡️ Guardrail Check ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Evaluation Started                                                                                   │
│  Name: def validate_log_analysis(result: TaskOutput) -> T...                                                    │
│  Status: 🔄 Evaluating                                                                                          │
│  Attempt: 2                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🛡️ Guardrail Success ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Passed                                                                                               │
│  Name: Validation Successful                                                                                    │
│  Status: ✅ Validated                                                                                           │
│  Attempts: 2                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Analyze the following log data to identify issues:                                                       │
│                                                                                                                 │
│  [2024-01-15 09:01:22.100] INFO: Cron job scheduled-cleanup started                                             │
│  [2024-01-15 09:01:23.200] INFO: Connected to database cluster (primary)                                        │
│  [2024-01-15 09:01:24.300] INFO: Processing batch 1 of 1                                                        │
│  [2024-01-15 09:01:25.400] INFO: 0 records processed                                                            │
│  [2024-01-15 09:01:26.500] INFO: Disk usage at 94%                                                              │
│  [2024-01-15 09:01:27.600] INFO: Cron job scheduled-cleanup completed successfully                              │
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d1e398ff-062d-45ae-8b18-88625d75b3fa                                                                       │
│  Final Output: {"primary_issue":"High disk usage reported during cleanup job","root_cause":"The logs indicate   │
│  that disk usage reached 94%, which may hinder database performance or future operations if the usage           │
│  continues to increase without cleanup.","errors":["Disk usage at 94%"],"affected_components":["Database        │
│  Cluster","Disk Storage"],"timeline":["2024-01-15 09:01:22.100 INFO: Cron job scheduled-cleanup                 │
│  started","2024-01-15 09:01:23.200 INFO: Connected to database cluster (primary)","2024-01-15 09:01:24.300      │
│  INFO: Processing batch 1 of 1","2024-01-15 09:01:25.400 INFO: 0 records processed","2024-01-15 09:01:26.500    │
│  INFO: Disk usage at 94%","2024-01-15 09:01:27.600 INFO: Cron job scheduled-cleanup completed successfully"]}   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [20]:
report = guarded_result.pydantic
print(f"Errors found: {len(report.errors)}")
for error in report.errors:
    print(f"  - {error}")
print(f"\nRoot cause: {report.root_cause}")

Errors found: 1
  - Disk usage at 94%

Root cause: The logs indicate that disk usage reached 94%, which may hinder database performance or future operations if the usage continues to increase without cleanup.


---

## 3. No-Code Guardrail

Writing validation functions for everything is tedious. For simpler checks, you can pass a **plain English string** as the guardrail. CrewAI uses an LLM to evaluate whether the output meets your criteria.

In [21]:
solution_specialist = Agent(
    role="DevOps Solution Specialist",
    goal="Provide clear, actionable solutions with step-by-step instructions",
    llm=llm,
    backstory="""You are a DevOps solutions architect who specializes in creating 
    reliable, step-by-step remediation plans for infrastructure issues.""",
    verbose=True,
)

solution_task = Task(
    description="Provide a solution for the following issue: {issue}",
    expected_output="A remediation plan with specific commands",
    guardrail="The solution must include at least 3 specific, copy-pasteable shell commands. "
    "Reject if it only contains general advice without concrete commands.",
    agent=solution_specialist,
)

solution_crew = Crew(
    agents=[solution_specialist],
    tasks=[solution_task],
    process=Process.sequential,
    verbose=True,
)

solution_result = await solution_crew.kickoff_async(
    inputs={"issue": "Kubernetes pods failing with ImagePullBackOff due to missing registry credentials"}
)
print(f"\nFinal solution:\n{solution_result.raw}")

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.13                                                                                       │
│  Latest version:  1.15.15                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a7ce4ddd-f608-41d0-bdf8-8e59929cbd54                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Provide a solution for the following issue: Kubernetes pods failing with ImagePullBackOff due to         │
│  missing registry credentials                                                                                   │
│  ID: 97ae29ad-cec6-45f9-ae02-faa99aa5693c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Solution Specialist                                                                              │
│                                                                                                                 │
│  Task: Provide a solution for the following issue: Kubernetes pods failing with ImagePullBackOff due to         │
│  missing registry credentials                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

╭────────────────────────────────────────────── 🛡️ Guardrail Success ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Passed                                                                                               │
│  Name: Validation Successful                                                                                    │
│  Status: ✅ Validated                                                                                           │
│  Attempts: 1                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Provide a solution for the following issue: Kubernetes pods failing with ImagePullBackOff due to         │
│  missing registry credentials                                                                                   │
│  Agent: DevOps Solution Specialist                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Final solution:
To remediate the issue of Kubernetes pods failing with `ImagePullBackOff` due to missing registry credentials, please follow the step-by-step plan below:

### Step 1: Identify the Problem
Use the following command to verify the pod's status and confirm the `ImagePullBackOff` error message.

```bash
kubectl describe pod <pod-name> -n <namespace>
```

Look for the message indicating that the image cannot be pulled due to missing credentials.

### Step 2: Create a Docker Registry Secret
You will need to create a Kubernetes secret with your Docker registry credentials. Run the following command, replacing placeholders with your actual values:

```bash
kubectl create secret docker-registry <secret-name> \
    --docker-server=<your-registry-server> \
    --docker-username=<your-username> \
    --docker-password=<your-password> \
    --docker-email=<your-email> \
    -n <namespace>
```

- `<secret-name>`: Name for your secret, e.g., `my-registry-secret`.
- `<your-registry-ser

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a7ce4ddd-f608-41d0-bdf8-8e59929cbd54                                                                       │
│  Final Output: To remediate the issue of Kubernetes pods failing with `ImagePullBackOff` due to missing         │
│  registry credentials, please follow the step-by-step plan below:                                               │
│                                                                                                                 │
│  ### Step 1: Identify the Problem                                                                               │
│  Use the following command to verify the pod's status and confirm the `ImagePullBackOff` error message.         │
│                                                                                                                 │
│  ```bash                                                                                                        │
│  kubectl describe pod <pod-name> -n <namespace>                                                                 │
│  ```                                                                                                            │
│                                                                                                                 │
│  Look for the message indicating that the image cannot be pulled due to missing credentials.                    │
│                                                                                                                 │
│  ### Step 2: Create a Docker Registry Secret                                                                    │
│  You will need to create a Kubernetes secret with your Docker registry credentials. Run the following command,  │
│  replacing placeholders with your actual values:                                                                │
│                                                                                                                 │
│  ```bash                                                                                                        │
│  kubectl create secret docker-registry <secret-name> \                                                          │
│      --docker-server=<your-registry-server> \                                                                   │
│      --docker-username=<your-username> \                                                                        │
│      --docker-password=<your-password> \                                                                        │
│      --docker-email=<your-email> \                                                                              │
│      -n <namespace>                                                                                             │
│  ```                                                                                                            │
│                                                                                                                 │
│  - `<secret-name>`: Name for your secret, e.g., `my-registry-secret`.                                           │
│  - `<your-registry-server>`: URL of your Docker registry (e.g., `https://index.docker.io/v1/` for Docker Hub).  │
│  - `<your-username>`: Your Docker registry username.                                                            │
│  - `<your-password>`: Your Docker registry password.                                                            │
│  - `<your-email>`: Your email address associated with 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---

## 4. The Full Pipeline

Everything from this notebook in one cell. Three agents, three tasks, every production feature working together:

- **Structured output** on the analysis task — typed `LogAnalysisReport` instead of raw text.
- **Code guardrail** on the analysis task — retries if no errors are identified.
- **Context passing** — each downstream task receives output from the tasks before it.
- **No-code guardrail** on the solution task — plain English check for concrete commands.
- **`output_file`** — all three task results saved to disk.

In [23]:
os.makedirs("task_outputs", exist_ok=True)

log_analyzer = Agent(
    role="DevOps Log Analyzer",
    goal="Analyze log files to identify and extract specific issues, errors, and failure patterns",
    llm=llm,
    backstory="""You are a senior DevOps engineer with 10 years of experience in 
    analyzing production logs and identifying critical issues. You excel at parsing 
    through complex log files, identifying error patterns, extracting relevant error 
    messages, and determining the root cause of failures from log data.""",
    verbose=True,
)

issue_investigator = Agent(
    role="DevOps Issue Investigator",
    goal="Investigate identified issues by searching documentation, forums, and known solutions",
    llm=llm,
    backstory="""You are a DevOps troubleshooting specialist who excels at quickly 
    finding solutions to technical problems. You know how to identify reliable sources 
    and gather comprehensive information about error patterns and their solutions.""",
    verbose=True,
)

solution_specialist = Agent(
    role="DevOps Solution Specialist",
    goal="Provide clear, actionable solutions with step-by-step instructions",
    llm=llm,
    backstory="""You are a DevOps solutions architect who specializes in creating 
    reliable, step-by-step remediation plans for infrastructure issues.""",
    verbose=True,
)

def validate_log_analysis(result: TaskOutput) -> Tuple[bool, Any]:
    report = result.pydantic
    if not report or not report.errors:
        return (False, "Must identify at least one error")
    return (True, report)

analyze_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="A structured log analysis report",
    output_pydantic=LogAnalysisReport,
    guardrail=validate_log_analysis,
    agent=log_analyzer,
    output_file="task_outputs/log_analysis.json",
)

investigate_task = Task(
    description="""Based on the log analysis findings, investigate the identified issue.
    
    Your investigation should:
    1. Identify common causes and scenarios for this type of issue
    2. Find known solutions and best practices
    3. Gather information about proven fixes and workarounds""",
    expected_output="""A comprehensive investigation report including:
    - Common causes ranked by likelihood
    - Known solutions and best practices
    - Recommended fixes and workarounds""",
    agent=issue_investigator,
    context=[analyze_task],
    output_file="task_outputs/investigation_report.md",
)

solution_task = Task(
    description="""Based on the log analysis and investigation findings, provide a complete solution.
    
    Your solution should:
    1. Create a step-by-step remediation plan with specific commands
    2. Provide verification steps to confirm the fix
    3. Suggest monitoring and prevention measures""",
    expected_output="A detailed remediation plan with step-by-step commands",
    guardrail="The solution must include at least 3 specific, copy-pasteable shell commands. "
    "Reject if it only contains general advice without concrete commands.",
    agent=solution_specialist,
    context=[analyze_task, investigate_task],
    output_file="task_outputs/solution_plan.md",
)

pipeline_crew = Crew(
    agents=[log_analyzer, issue_investigator, solution_specialist],
    tasks=[analyze_task, investigate_task, solution_task],
    process=Process.sequential,
    verbose=True,
)

result = await pipeline_crew.kickoff_async(inputs={"log_data": LOG_INPUT})

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.13                                                                                       │
│  Latest version:  1.15.15                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: cf623bd9-747a-4776-a0ba-e58644bfe14f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following log data to identify issues:                                                       │
│                                                                                                                 │
│  [2024-01-15 14:32:15.123] INFO: Starting deployment of myapp-deployment                                        │
│  [2024-01-15 14:32:16.567] WARNING: Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state                       │
│  [2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start                          │
│  [2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not  │
│  exist or may require 'docker login'                                                                            │
│  [2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff                 │
│  [2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its         │
│  progress deadline                                                                                              │
│  [2024-01-15 14:32:26.789] WARNING: Service myapp-service has no available endpoints                            │
│  [2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated                          │
│                                                                                                                 │
│  ID: a68a3e06-3ba7-4e2a-aae5-7890d8ac3d6d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Task: Analyze the following log data to identify issues:                                                       │
│                                                                                                                 │
│  [2024-01-15 14:32:15.123] INFO: Starting deployment of myapp-deployment                                        │
│  [2024-01-15 14:32:16.567] WARNING: Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state                       │
│  [2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start                          │
│  [2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not  │
│  exist or may require 'docker login'                                                                            │
│  [2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff                 │
│  [2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its         │
│  progress deadline                                                                                              │
│  [2024-01-15 14:32:26.789] WARNING: Service myapp-service has no available endpoints                            │
│  [2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

╭──────────────────────────────────────────── ✅ LiteAgent Completed ─────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Completed                                                                                            │
│  Role: Guardrail Agent                                                                                          │
│  id: a796a324-2a2d-4868-b4bb-a53abcfb6e5a                                                                       │
│  role: Guardrail Agent                                                                                          │
│  goal: Validate the output of the task                                                                          │
│  backstory: You are a expert at validating the output of a task. By providing effective feedback if the output  │
│  is not valid.                                                                                                  │
│  tools: None                                                                                                    │
│  verbose: False                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🛡️ Guardrail Success ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Passed                                                                                               │
│  Name: Validation Successful                                                                                    │
│  Status: ✅ Validated                                                                                           │
│  Attempts: 1                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on the log analysis and investigation findings, provide a complete solution.                       │
│                                                                                                                 │
│      Your solution should:                                                                                      │
│      1. Create a step-by-step remediation plan with specific commands                                           │
│      2. Provide verification steps to confirm the fix                                                           │
│      3. Suggest monitoring and prevention measures                                                              │
│  Agent: DevOps Solution Specialist                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: cf623bd9-747a-4776-a0ba-e58644bfe14f                                                                       │
│  Final Output: # Remediation Plan for Production Deployment Failure Due to Image Pull Errors                    │
│                                                                                                                 │
│  ## 1. Step-by-Step Remediation Plan                                                                            │
│                                                                                                                 │
│  ### Step 1: Verify Image Availability                                                                          │
│  First, confirm that the specified image (`myapp:v1.2.3`) exists in the container registry.                     │
│                                                                                                                 │
│  1. **Login to the Container Registry**:                                                                        │
│     ```bash                                                                                                     │
│     docker login <registry-url>                                                                                 │
│     ```                                                                                                         │
│                                                                                                                 │
│  2. **Check If Image Exists**:                                                                                  │
│     Attempt to pull the image manually to see if there are issues.                                              │
│     ```bash                                                                                                     │
│     docker pull <registry-url>/myapp:v1.2.3                                                                     │
│     ```                                                                                                         │
│                                                                                                                 │
│  ### Step 2: Create Image Pull Secret (if necessary)                                                            │
│  If your image is stored in a private repository, ensure that you create an image pull secret if one doesn't    │
│  exist.                                                                                                         │
│                                                                                                                 │
│  3. **Create the Secret**:                                                                                      │
│     ```bash                                                                                                     │
│     kubectl create secret docker-registry myregistrykey --docker-username=<your-username>                       │
│  --docker-password=<your-password> --docker-email=<your-email>                                                  │
│     ```                                                                                                         │
│                                                                                                                 │
│  ### Step 3: Update the Deployment Configuration                                                                │
│  Update your deployment YAML to reference the created 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
report = analyze_task.output.pydantic
print("STRUCTURED ANALYSIS (from guardrailed task):")
print(f"  Primary issue: {report.primary_issue}")
print(f"  Root cause: {report.root_cause}")
print(f"  Errors: {report.errors}")
print(f"  Affected components: {report.affected_components}")
print(f"  Timeline: {report.timeline}")

print(f"\nSOLUTION (from no-code guardrailed task):")
print(result.raw)
print(result.raw)
                "print(result.raw)",
                "",
                "# --- Streamlit dashboard setup (writes app file + launches it) ---",
                "from pathlib import Path",
                "import json, subprocess, sys, time, os",
                "",
                "app_path = Path('agents/intermediate/v2/streamlit_dashboard.py')",
                "app_path.parent.mkdir(parents=True, exist_ok=True)",
                "app_code = '''import streamlit as st\nimport json\nfrom pathlib import Path\nst.set_page_config(page_title=\"CrewAI Pipeline Dashboard\", layout=\"wide\")\nst.title(\"CrewAI Pipeline Dashboard\")\ndata_dir = Path(\"task_outputs\")\nlog_file = data_dir / \"log_analysis.json\"\ninvestigate_file = data_dir / \"investigation_report.md\"\nsolution_file = data_dir / \"solution_plan.md\"\ncols = st.columns([1,2])\nwith cols[0]:\n    st.subheader(\"Pipeline Flow\")\n    dot = \"digraph G { rankdir=LR; Analyze -> Investigate -> Solution; Analyze[label=\\\"Analyze\\\"]; Investigate[label=\\\"Investigate\\\"]; Solution[label=\\\"Solution\\\"]; }\"\n    st.graphviz_chart(dot)\nwith cols[1]:\n    st.subheader(\"Outputs\")\n    if log_file.exists():\n        try:\n            j = json.loads(log_file.read_text(encoding='utf-8'))\n            st.write(\"### Log Analysis (structured)\")\n            st.json(j)\n        except Exception as e:\n            st.write(\"Could not read JSON:\", e)\n    else:\n        st.info(\"Run the pipeline to produce task_outputs/log_analysis.json\")\n    st.write(\"---\")\n    if investigate_file.exists():\n        st.write(\"### Investigation (markdown)\")\n        st.code(investigate_file.read_text(encoding='utf-8'), language='markdown')\n    if solution_file.exists():\n        st.write(\"### Solution (markdown)\")\n        st.code(solution_file.read_text(encoding='utf-8'), language='markdown')\n'''",
                "app_path.write_text(app_code, encoding='utf-8')",
                "",
                "# Ensure streamlit is installed and then attempt to launch the app in background",
                "try:",
                "    import streamlit  # type: ignore",
                "except Exception:",
                "    print('Installing streamlit...')",
                "    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'streamlit'])",
                "",
                "cmd = [sys.executable, '-m', 'streamlit', 'run', str(app_path), '--server.port=8501', '--server.headless=true']",
                "print('Streamlit command:', ' '.join(cmd))",
                "try:",
                "    if os.name == 'nt':",
                "        p = subprocess.Popen(cmd, creationflags=0x00000200)",
                "    else:",
                "        p = subprocess.Popen(cmd)",
                "    time.sleep(1)",
                "    print(f\"Streamlit PID: {p.pid}\")",
                "    print('Open http://localhost:8501 to view the dashboard.')",
                "except Exception as e:",
                "    print('Failed to start Streamlit automatically:', e)",
                "    print('Run the command above in a terminal.')"

STRUCTURED ANALYSIS (from guardrailed task):
  Primary issue: Production deployment failed due to image pull errors.
  Root cause: The pod could not start because it was unable to pull the required image 'myapp:v1.2.3', either due to lack of access or because the repository does not exist. This led to an 'ImagePullBackOff' status and the deployment exceeding its progress deadline.
  Errors: ['Pod myapp-deployment-7b8c9d5f4-abc12 failed to start', 'Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require \'docker login\'', 'Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff', 'Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline', 'Production deployment failed - rollback initiated']
  Affected components: ['myapp-deployment', 'myapp-service']
  Timeline: ['[2024-01-15 14:32:15.123] INFO: Starting deployment of myapp-deployment', '[2024-01-15 14:32:16.567] WARNING: Pod myapp-deployment-7b8c9d5f4-abc12 

---

## Recap

| Feature | Without (Notebook 1) | With (This Notebook) | One-Line Change |
|---------|---------------------|---------------------|----------------|
| **Structured Output** | Raw markdown text, fragile parsing | Typed Python objects, guaranteed schema | `output_pydantic=Model` |
| **Code Guardrail** | Bad output passes through silently | Agent retries until output is valid | `guardrail=validate_fn` |
| **No-Code Guardrail** | Write validation code for every check | Describe the check in plain English | `guardrail="must have 3 commands"` |
| **Full Pipeline** | Features used in isolation | All features combined in one crew | Section 4 |